# SOXL/SOXX Target Relative-Wealth LFNN

## 假设与边界

按论文的 global-in-time 思路，用可微财富递推直接优化三年末策略财富相对 100% SOXX 的目标，而不是预测单日涨跌或使用强化学习奖励。网络每 63 个交易日输出 SOXL、SOXX、cash 的 long-only softmax 权重；训练只用 2011–2019，2020–2022 用于 bootstrap validation，2023–2026 outer 只做已观察样本的诊断。

当前 qlib store 没有 PIT 债券/现金收益，cash 固定为零；SOXL/SOXX 是集中半导体映射，也不等同于论文中的宽基 LETF/VETF。

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd().resolve()
ROOT = next(path for path in (HERE, *HERE.parents) if (path / 'src').is_dir() and (path / 'SPEC.md').exists())
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

import qlib
from qlib.config import REG_US
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R

PROVIDER = ROOT / 'data' / 'qlib' / 'soxl'
qlib.init(provider_uri=PROVIDER.as_posix(), region=REG_US)
calendar = D.calendar(start_time='2011-01-01', end_time='2026-12-31', freq='day')

from src.workflow.relative_wealth_lfnn_soxx import run_relative_wealth_lfnn_soxx_card

CARD = ROOT / 'research' / 'relative_wealth_lfnn_soxx_v1'
summary = run_relative_wealth_lfnn_soxx_card(root=ROOT)
display({
    'calendar_rows': len(calendar),
    'recorder_id': summary['recorder_id'],
    'as_of': summary['data_audit']['as_of'],
    'promotion_decision': summary['promotion_decision'],
    'selection_uses_outer_test': summary['selection_uses_outer_test'],
})

[22367:MainThread](2026-07-22 11:02:21,043) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[22367:MainThread](2026-07-22 11:02:21,044) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[22367:MainThread](2026-07-22 11:02:21,044) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[22367:MainThread](2026-07-22 11:02:21,825) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[22367:MainThread](2026-07-22 11:02:21,858) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[22367:MainThread](2026-07-22 11:02:37,541) INFO - qlib.workflow - [exp.py:258] - Experiment 890311729946472326 starts running ...


[22367:MainThread](2026-07-22 11:02:37,605) INFO - qlib.workflow - [recorder.py:345] - Recorder dbd71f0291ba4ac680198c4cfc06341c starts running under Experiment 890311729946472326 ...


[22367:MainThread](2026-07-22 11:02:37,720) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'calendar_rows': 3908,
 'recorder_id': 'dbd71f0291ba4ac680198c4cfc06341c',
 'as_of': '2026-07-20',
 'promotion_decision': 'research_reject_relative_wealth_validation_distribution_gates_failed',
 'selection_uses_outer_test': False}

## 未见 outer 的模型选择与 bootstrap validation

In [2]:
model_selection = pd.read_csv(CARD / summary['sidecars']['model_selection'])
bootstrap = pd.read_csv(CARD / summary['sidecars']['bootstrap_metrics'])
display(model_selection[[
    'variant', 'annual_excess_target', 'median_ratio_10bps',
    'outperformance_probability_10bps', 'active_ir_10bps',
    'positive_seeds_10bps', 'validation_gate_pass',
]])
display(bootstrap[[
    'cost_bps', 'mean_terminal_wealth_ratio', 'median_terminal_wealth_ratio',
    'outperformance_probability', 'active_information_ratio',
    'avg_soxl_weight', 'avg_soxx_weight', 'avg_cash_weight',
]])

,variant,annual_excess_target,median_ratio_10bps,outperformance_probability_10bps,active_ir_10bps,positive_seeds_10bps,validation_gate_pass
0,market_state_augmented,0.06,0.859672,0.254883,-0.689567,0,False
1,paper_minimal,0.06,0.860667,0.229736,-0.772272,0,False
2,market_state_augmented,0.04,0.870168,0.218750,-0.800309,0,False
3,paper_minimal,0.04,0.878085,0.199707,-0.857186,0,False
4,market_state_augmented,0.02,0.881036,0.122070,-1.136130,0,False
5,paper_minimal,0.02,0.881244,0.090820,-1.238433,0,False


,cost_bps,mean_terminal_wealth_ratio,median_terminal_wealth_ratio,outperformance_probability,active_information_ratio,avg_soxl_weight,avg_soxx_weight,avg_cash_weight
0,10,0.880399,0.861601,0.244141,-0.720005,0.311293,0.354158,0.334548
1,25,0.879579,0.860748,0.243652,-0.724428,0.311308,0.354178,0.334514


## Outer 诊断：不参与选择

In [3]:
comparison = pd.read_csv(CARD / summary['sidecars']['backtest_benchmark_comparison'])
annual = pd.read_csv(CARD / summary['sidecars']['annual_outer_metrics'])
display(comparison[[
    'strategy', 'cost_bps', 'return_pct', 'terminal_wealth_ratio_vs_soxx',
    'active_information_ratio_daily', 'max_drawdown_pct', 'worst_day_pct',
    'cvar5_pct', 'worst_rolling_5d_pct', 'avg_soxl_position_pct',
]])
display(annual.loc[annual['cost_bps'].eq(10)])

,strategy,cost_bps,return_pct,terminal_wealth_ratio_vs_soxx,active_information_ratio_daily,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,avg_soxl_position_pct
0,relative_wealth_lfnn_ensemble,10,523.142999,1.366823,0.924707,-48.361701,-13.629603,-6.706384,-20.714360,32.740178
1,soxx_buyhold,10,355.906272,1.000000,0.000000,-43.091932,-9.654278,-5.179665,-17.688143,0.000000
2,one_third_soxl_cash,10,291.701872,0.859172,-1.833323,-45.796223,-9.108204,-5.083038,-17.930467,33.333333
3,half_soxl_cash,10,549.809931,1.425315,1.003989,-61.716117,-13.662305,-7.624557,-26.119561,50.000000
4,relative_wealth_lfnn_ensemble,25,521.201739,1.364620,0.921637,-48.374938,-13.630246,-6.706806,-20.715445,32.742074
5,soxx_buyhold,25,355.219418,1.000000,0.000000,-43.091932,-9.654278,-5.179665,-17.688143,0.000000
6,one_third_soxl_cash,25,291.505426,0.860037,-1.819684,-45.796223,-9.108204,-5.083038,-17.930467,33.333333
7,half_soxl_cash,25,549.320347,1.426390,1.005159,-61.716117,-13.662305,-7.624557,-26.119561,50.000000


,strategy,cost_bps,outer_year,return_pct,cagr_pct,annualized_vol_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,...,avg_equity_position_pct,avg_effective_beta,upside_capture_pct,calmar,total_turnover,qlib_annualized_return_pct,qlib_information_ratio,qlib_max_drawdown_pct,rows,soxx_return_pct
0,relative_wealth_lfnn_ensemble,10,2023,66.876193,67.561229,39.799810,-23.740221,-6.426079,-4.714958,-8.439137,...,68.780021,1.365001,45.279819,2.845855,0.919107,67.561229,1.306394,-23.740221,250,55.199245
1,relative_wealth_lfnn_ensemble,10,2024,17.995078,17.995078,39.300410,-29.559889,-9.345596,-5.941805,-16.015072,...,63.593062,1.227440,40.776148,0.608767,0.361710,17.995078,0.418501,-29.559889,252,20.020878
2,relative_wealth_lfnn_ensemble,10,2025,80.695557,81.552843,47.373221,-35.829860,-8.778347,-6.274955,-14.671978,...,65.569483,1.289359,42.959997,2.276114,0.617695,81.552843,1.265306,-35.829860,250,46.941506
3,relative_wealth_lfnn_ensemble,10,2026,75.138640,186.882074,76.049974,-35.268189,-13.629603,-9.894477,-20.714360,...,69.732591,1.468813,49.157485,5.298885,0.223101,186.882074,1.387852,-35.268189,134,66.565356
4,soxx_buyhold,10,2023,55.199245,55.745936,28.867502,-17.994919,-5.119705,-3.360507,-6.594465,...,100.000000,1.000000,33.417002,3.097871,1.000000,55.745936,1.543353,-17.994919,250,55.199245
5,soxx_buyhold,10,2024,20.020878,20.020878,32.616461,-27.722259,-9.654278,-4.974844,-16.627814,...,100.000000,1.000000,33.606485,0.722195,0.000000,20.020878,0.555311,-27.722259,252,20.020878
6,soxx_buyhold,10,2025,46.941506,47.394623,39.684357,-35.438265,-8.806676,-5.782489,-17.688143,...,100.000000,1.000000,34.081656,1.337386,0.000000,47.394623,0.979240,-35.438265,250,46.941506
7,soxx_buyhold,10,2026,66.565356,161.042658,49.386623,-22.235301,-7.083005,-6.037083,-11.602696,...,100.000000,1.000000,33.591423,7.242657,0.000000,161.042658,1.949327,-22.235301,134,66.565356
8,one_third_soxl_cash,10,2023,50.869681,51.366855,28.973319,-18.268515,-4.953257,-3.336050,-6.748411,...,33.333333,1.000000,33.333335,2.811770,0.333333,51.366855,1.439219,-18.268515,250,55.199245
9,one_third_soxl_cash,10,2024,14.131473,14.131473,32.683070,-28.732424,-9.108204,-4.890567,-16.942511,...,33.333333,1.000000,33.333333,0.491830,0.000000,14.131473,0.401946,-28.732424,252,20.020878


## 最终中文结论

In [4]:
recorder = R.get_recorder(
    recorder_id=summary['recorder_id'],
    experiment_name=summary['experiment'],
)
display({
    'recorder_local_dir': recorder.get_local_dir(),
    'recorder_only_artifacts': summary['artifact_policy']['recorder_only_artifacts'],
})
display(Markdown((CARD / 'conclusion.md').read_text(encoding='utf-8')))

{'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/relative_wealth_lfnn_soxx_v1/qlib_recorder/890311729946472326/dbd71f0291ba4ac680198c4cfc06341c',
 'recorder_only_artifacts': ['artifacts/relative_wealth_lfnn_soxx_v1/models/',
  'artifacts/relative_wealth_lfnn_soxx_v1/training_history.csv',
  'artifacts/relative_wealth_lfnn_soxx_v1/bootstrap_terminal_wealth.csv',
  'artifacts/relative_wealth_lfnn_soxx_v1/diagnostic_seed0_terminal_wealth.csv',
  'artifacts/relative_wealth_lfnn_soxx_v1/positions.csv',
  'artifacts/relative_wealth_lfnn_soxx_v1/strategy_returns.csv']}

# 中文结论

## 跑了什么

用 qlib DatasetH/DataHandlerLP 读取 SOXL/SOXX 因果状态与 next-open returns，在 2011–2019 上训练 paper-inspired global-in-time feasible network。模型直接输出 SOXL/SOXX/cash long-only 权重，以 100% SOXX 为基准，按 63 交易日调仓，并在 stationary block bootstrap 三年路径上优化相对终值目标。2020–2022 只用于 validation；2023–2026-07 只做已经看过的诊断 outer。

## Bootstrap validation

选中 `market_state_augmented`、年化目标 6.0%。10bp：终值财富比中位数 0.8616，跑赢概率 24.41%，Active IR -0.7200。25bp：中位数 0.8607，跑赢概率 24.37%，Active IR -0.7244。validation gate=False。

## Outer诊断

10bp 策略收益 523.14%，SOXX 355.91%，终值财富比 1.3668，Active IR 0.9247，MaxDD -48.36%。

## 决定

`research_reject_relative_wealth_validation_distribution_gates_failed`。无论 outer 是否好看，这个架构是在 outer 已被观察后提出，不能据此晋升；需要 2026-07-21 之后 sealed forward。当前 cash 收益按 0，且 SOXL/SOXX 是集中半导体映射，不等同论文宽基 LETF+债券。

不构成买卖建议。
